<a href="https://colab.research.google.com/github/swathi-men1/AI_TransferIQ/blob/main/milestone1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pandas as pd, warnings
warnings.filterwarnings('ignore')

folder = '/content/drive/MyDrive/TransferIQ Project'
os.makedirs(folder, exist_ok=True)
print("Ready!")

In [ ]:
!pip install statsbombpy --quiet
from statsbombpy import sb
import pandas as pd

# Load ALL free competitions
comps = sb.competitions()
print(f"Available competitions: {len(comps)}")

# Collect matches from multiple competitions for large dataset
competition_list = [
    (11, 27),   # La Liga 2015/16
    (11, 26),   # La Liga 2014/15
    (11, 25),   # La Liga 2013/14
    (2,  44),   # Premier League 2003/04
    (37, 42),   # FA Women's Super League
]

all_shots, all_passes, all_dribbles = [], [], []

for comp_id, season_id in competition_list:
    try:
        matches = sb.matches(competition_id=comp_id, season_id=season_id)
        print(f"Competition {comp_id}/{season_id}: {len(matches)} matches")

        for mid in matches['match_id']:
            events = sb.events(match_id=mid)

            shots    = events[events['type']=='Shot'][['player','team','shot_outcome','shot_statsbomb_xg']].copy()
            passes   = events[events['type']=='Pass'][['player','team','pass_outcome']].copy()
            dribbles = events[events['type']=='Dribble'][['player','team','dribble_outcome']].copy()

            for df in [shots, passes, dribbles]:
                df['match_id'] = mid
                df['competition'] = comp_id

            all_shots.append(shots)
            all_passes.append(passes)
            all_dribbles.append(dribbles)
    except:
        print(f"Skipping {comp_id}/{season_id}")

df_shots    = pd.concat(all_shots,    ignore_index=True)
df_passes   = pd.concat(all_passes,   ignore_index=True)
df_dribbles = pd.concat(all_dribbles, ignore_index=True)

print(f"\nTotal shots:    {len(df_shots):,}")
print(f"Total passes:   {len(df_passes):,}")
print(f"Total dribbles: {len(df_dribbles):,}")

In [ ]:
# Merge players with their full valuation history
df_market = pd.merge(
    df_players[['player_id','name','position','market_value_in_eur',
                'current_club_name','country_of_citizenship','date_of_birth']],
    df_valuations[['player_id','date','market_value_in_eur']],
    on='player_id', how='left'
)
print(f"Market dataset: {df_market.shape}")

df_market.to_csv('/content/drive/MyDrive/TransferIQ Project/market_values.csv', index=False)
df_transfers.to_csv('/content/drive/MyDrive/TransferIQ Project/transfer_history.csv', index=False)
print("Saved market values + transfer history!")

In [ ]:
# Aggregate into per-player stats
player_stats = df_shots.groupby(['player','team']).agg(
    total_shots  = ('shot_statsbomb_xg', 'count'),
    goals        = ('shot_outcome', lambda x: (x=='Goal').sum()),
    total_xg     = ('shot_statsbomb_xg', 'sum'),
    avg_xg_shot  = ('shot_statsbomb_xg', 'mean')
).reset_index()

pass_stats = df_passes.groupby('player').agg(
    total_passes     = ('pass_outcome', 'count'),
    successful_passes= ('pass_outcome', lambda x: x.isna().sum())
).reset_index()

dribble_stats = df_dribbles.groupby('player').agg(
    total_dribbles     = ('dribble_outcome', 'count'),
    successful_dribbles= ('dribble_outcome', lambda x: (x=='Complete').sum())
).reset_index()

from functools import reduce
df_performance = reduce(
    lambda l,r: pd.merge(l, r, on='player', how='outer'),
    [player_stats, pass_stats, dribble_stats]
).fillna(0)

print(f"Player performance dataset: {df_performance.shape}")
df_performance.to_csv('/content/drive/MyDrive/TransferIQ Project/player_performance.csv', index=False)
print("Saved!")

In [ ]:
base = 'https://raw.githubusercontent.com/dcaribou/transfermarkt-datasets/master/data'

df_players    = pd.read_csv(f'{base}/players.csv')
df_valuations = pd.read_csv(f'{base}/player_valuations.csv')
df_clubs      = pd.read_csv(f'{base}/clubs.csv')
df_transfers  = pd.read_csv(f'{base}/transfers.csv')

print(f"Players:    {df_players.shape}")
print(f"Valuations: {df_valuations.shape}")
print(f"Clubs:      {df_clubs.shape}")
print(f"Transfers:  {df_transfers.shape}")

print("\nKey columns in players:")
print(df_players[['name','position','market_value_in_eur',
                   'highest_market_value_in_eur','current_club_name',
                   'country_of_citizenship','date_of_birth']].head(5))

In [ ]:
base = 'https://raw.githubusercontent.com/dcaribou/transfermarkt-datasets/master/data'

df_players    = pd.read_csv(f'{base}/players.csv')
df_valuations = pd.read_csv(f'{base}/player_valuations.csv')
df_clubs      = pd.read_csv(f'{base}/clubs.csv')
df_transfers  = pd.read_csv(f'{base}/transfers.csv')

print(f"Players:    {df_players.shape}")
print(f"Valuations: {df_valuations.shape}")
print(f"Clubs:      {df_clubs.shape}")
print(f"Transfers:  {df_transfers.shape}")

print("\nKey columns in players:")
print(df_players[['name','position','market_value_in_eur',
                   'highest_market_value_in_eur','current_club_name',
                   'country_of_citizenship','date_of_birth']].head(5))

In [ ]:
# New CDN base URL — this is the updated working link
base = 'https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data'

print("Loading players...")
df_players = pd.read_csv(f'{base}/players.csv.gz', compression='gzip')
print(f"Players: {df_players.shape}")

print("Loading valuations...")
df_valuations = pd.read_csv(f'{base}/player_valuations.csv.gz', compression='gzip')
print(f"Valuations: {df_valuations.shape}")

print("Loading transfers...")
df_transfers = pd.read_csv(f'{base}/transfers.csv.gz', compression='gzip')
print(f"Transfers: {df_transfers.shape}")

print("Loading appearances...")
df_appearances = pd.read_csv(f'{base}/appearances.csv.gz', compression='gzip')
print(f"Appearances: {df_appearances.shape}")

print("Loading clubs...")
df_clubs = pd.read_csv(f'{base}/clubs.csv.gz', compression='gzip')
print(f"Clubs: {df_clubs.shape}")

print("Loading games...")
df_games = pd.read_csv(f'{base}/games.csv.gz', compression='gzip')
print(f"Games: {df_games.shape}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, os
folder = '/content/drive/MyDrive/TransferIQ Project'
os.makedirs(folder, exist_ok=True)

In [ ]:
# New CDN base URL — this is the updated working link
base = 'https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data'

print("Loading players...")
df_players = pd.read_csv(f'{base}/players.csv.gz', compression='gzip')
print(f"Players: {df_players.shape}")

print("Loading valuations...")
df_valuations = pd.read_csv(f'{base}/player_valuations.csv.gz', compression='gzip')
print(f"Valuations: {df_valuations.shape}")

print("Loading transfers...")
df_transfers = pd.read_csv(f'{base}/transfers.csv.gz', compression='gzip')
print(f"Transfers: {df_transfers.shape}")

print("Loading appearances...")
df_appearances = pd.read_csv(f'{base}/appearances.csv.gz', compression='gzip')
print(f"Appearances: {df_appearances.shape}")

print("Loading clubs...")
df_clubs = pd.read_csv(f'{base}/clubs.csv.gz', compression='gzip')
print(f"Clubs: {df_clubs.shape}")

print("Loading games...")
df_games = pd.read_csv(f'{base}/games.csv.gz', compression='gzip')
print(f"Games: {df_games.shape}")

In [ ]:
base = 'https://raw.githubusercontent.com/dcaribou/transfermarkt-datasets/master/data'

df_players    = pd.read_csv(f'{base}/players.csv')
df_valuations = pd.read_csv(f'{base}/player_valuations.csv')
df_clubs      = pd.read_csv(f'{base}/clubs.csv')
df_transfers  = pd.read_csv(f'{base}/transfers.csv')

print(f"Players:    {df_players.shape}")
print(f"Valuations: {df_valuations.shape}")
print(f"Clubs:      {df_clubs.shape}")
print(f"Transfers:  {df_transfers.shape}")

print("\nKey columns in players:")
print(df_players[['name','position','market_value_in_eur',
                   'highest_market_value_in_eur','current_club_name',
                   'country_of_citizenship','date_of_birth']].head(5))

In [ ]:
import requests
from bs4 import BeautifulSoup
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import time

!pip install vaderSentiment --quiet

analyzer = SentimentIntensityAnalyzer()
headers  = {'User-Agent': 'Mozilla/5.0 Chrome/120.0.0.0'}

def scrape_headlines(url, source_name):
    r    = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.content, 'html.parser')
    headlines = []
    for tag in soup.find_all(['h1','h2','h3']):
        text = tag.get_text(strip=True)
        if len(text) > 25:
            headlines.append({'text': text, 'source': source_name})
    return headlines

sources = [
    ('https://www.bbc.com/sport/football',        'BBC Sport'),
    ('https://www.skysports.com/transfer-centre', 'Sky Sports'),
    ('https://www.theguardian.com/football',      'Guardian'),
    ('https://www.goal.com/en-gb/transfer-news',  'Goal.com'),
]

all_headlines = []
for url, name in sources:
    try:
        h = scrape_headlines(url, name)
        all_headlines.extend(h)
        print(f"{name}: {len(h)} headlines")
        time.sleep(1)
    except Exception as e:
        print(f"{name}: failed — {e}")

# Run sentiment
results = []
for item in all_headlines:
    scores = analyzer.polarity_scores(item['text'])
    results.append({
        'text':     item['text'],
        'source':   item['source'],
        'compound': scores['compound'],
        'positive': scores['pos'],
        'negative': scores['neg'],
        'neutral':  scores['neu'],
        'label':    'positive' if scores['compound'] >= 0.05
                    else ('negative' if scores['compound'] <= -0.05 else 'neutral')
    })

df_sentiment = pd.DataFrame(results)
print(f"\nTotal sentiment records: {len(df_sentiment)}")
print(df_sentiment['label'].value_counts())

df_sentiment.to_csv('/content/drive/MyDrive/TransferIQ Project/sentiment_data.csv', index=False)
print("Saved!")

In [ ]:
# Cell 1 — Install first
!pip install vaderSentiment --quiet
print("Installed!")

In [ ]:
import requests
from bs4 import BeautifulSoup
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import time

!pip install vaderSentiment --quiet

analyzer = SentimentIntensityAnalyzer()
headers  = {'User-Agent': 'Mozilla/5.0 Chrome/120.0.0.0'}

def scrape_headlines(url, source_name):
    r    = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.content, 'html.parser')
    headlines = []
    for tag in soup.find_all(['h1','h2','h3']):
        text = tag.get_text(strip=True)
        if len(text) > 25:
            headlines.append({'text': text, 'source': source_name})
    return headlines

sources = [
    ('https://www.bbc.com/sport/football',        'BBC Sport'),
    ('https://www.skysports.com/transfer-centre', 'Sky Sports'),
    ('https://www.theguardian.com/football',      'Guardian'),
    ('https://www.goal.com/en-gb/transfer-news',  'Goal.com'),
]

all_headlines = []
for url, name in sources:
    try:
        h = scrape_headlines(url, name)
        all_headlines.extend(h)
        print(f"{name}: {len(h)} headlines")
        time.sleep(1)
    except Exception as e:
        print(f"{name}: failed — {e}")

# Run sentiment
results = []
for item in all_headlines:
    scores = analyzer.polarity_scores(item['text'])
    results.append({
        'text':     item['text'],
        'source':   item['source'],
        'compound': scores['compound'],
        'positive': scores['pos'],
        'negative': scores['neg'],
        'neutral':  scores['neu'],
        'label':    'positive' if scores['compound'] >= 0.05
                    else ('negative' if scores['compound'] <= -0.05 else 'neutral')
    })

df_sentiment = pd.DataFrame(results)
print(f"\nTotal sentiment records: {len(df_sentiment)}")
print(df_sentiment['label'].value_counts())

df_sentiment.to_csv('/content/drive/MyDrive/TransferIQ Project/sentiment_data.csv', index=False)
print("Saved!")

In [ ]:
injury_url = 'https://raw.githubusercontent.com/salimt/football-datasets/main/datalake/transfermarkt/raw/player_injury_histories/player_injury_histories.csv'

df_injuries = pd.read_csv(injury_url)
print(f"Injury records: {df_injuries.shape}")
print(df_injuries.columns.tolist())
print(df_injuries.head(5))

df_injuries.to_csv('/content/drive/MyDrive/TransferIQ Project/injury_history.csv', index=False)
print("Saved!")

In [ ]:
!pip install duckdb --quiet

from google.colab import drive
drive.mount('/content/drive')

import duckdb, pandas as pd, os
folder = '/content/drive/MyDrive/TransferIQ Project'
os.makedirs(folder, exist_ok=True)
print("Ready!")

In [ ]:
# Official DuckDB query method — works 100%
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

base = 'https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data'

def load_tm(table):
    print(f"Loading {table}...")
    df = con.execute(f"""
        SELECT * FROM read_csv_auto('{base}/{table}.csv.gz')
    """).df()
    print(f"  Done: {df.shape[0]:,} rows x {df.shape[1]} cols")
    return df

df_players     = load_tm('players')
df_valuations  = load_tm('player_valuations')
df_transfers   = load_tm('transfers')
df_appearances = load_tm('appearances')
df_clubs       = load_tm('clubs')
df_games       = load_tm('games')

In [ ]:
print("=== PLAYERS (sample) ===")
print(df_players[['player_id','name','position',
                   'market_value_in_eur','current_club_name',
                   'country_of_citizenship','date_of_birth']].head(5))

print("\n=== VALUATIONS (sample) ===")
print(df_valuations[['player_id','date',
                      'market_value_in_eur']].head(5))

print("\n=== TRANSFERS (sample) ===")
print(df_transfers[['player_id','transfer_date','from_club_name',
                     'to_club_name','transfer_fee']].head(5))

print("\n=== APPEARANCES (sample) ===")
print(df_appearances[['player_id','game_id','goals',
                       'assists','minutes_played']].head(5))

In [ ]:
datasets = {
    'players.csv':           df_players,
    'market_valuations.csv': df_valuations,
    'transfer_history.csv':  df_transfers,
    'appearances.csv':       df_appearances,
    'clubs.csv':             df_clubs,
    'games.csv':             df_games,
}

print(f"{'File':<28} {'Rows':>10} {'Cols':>6} {'Size':>10}")
print("-" * 58)

for fname, df in datasets.items():
    df.to_csv(f'{folder}/{fname}', index=False)
    size = os.path.getsize(f'{folder}/{fname}') / 1024
    print(f"{fname:<28} {len(df):>10,} {df.shape[1]:>6} {size:>8.1f} KB")

print("\nAll saved to Drive!")

In [ ]:
# This one loads directly via pandas — no issues
transfers_url = 'https://raw.githubusercontent.com/d2ski/football-transfers-data/master/dataset/transfers.csv'

df_transfer_fees = pd.read_csv(transfers_url)
print(f"Transfer fees dataset: {df_transfer_fees.shape}")
print(df_transfer_fees[['player_name','player_age','player_pos',
                          'transfer_fee_amnt','market_val_amnt','season']].head(10))

df_transfer_fees.to_csv(f'{folder}/transfer_fees.csv', index=False)
print("Saved transfer_fees.csv!")

In [ ]:
!pip install requests beautifulsoup4 lxml --quiet

from google.colab import drive
drive.mount('/content/drive')

import requests, time, pandas as pd, os
from bs4 import BeautifulSoup

folder  = '/content/drive/MyDrive/TransferIQ Project'
os.makedirs(folder, exist_ok=True)
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'}
print("Ready!")

In [ ]:
def get_injuries(player_id, player_name):
    url  = f'https://www.transfermarkt.com/{player_name}/verletzungen/spieler/{player_id}'
    r    = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.content, 'lxml')
    rows = soup.select('table.items tbody tr')

    injuries = []
    for row in rows:
        cols = row.find_all('td')
        if len(cols) < 5:
            continue
        injuries.append({
            'player_id':     player_id,
            'player_name':   player_name.replace('-', ' ').title(),
            'season':        cols[0].get_text(strip=True),
            'injury':        cols[1].get_text(strip=True),
            'from_date':     cols[2].get_text(strip=True),
            'until_date':    cols[3].get_text(strip=True),
            'days_missed':   cols[4].get_text(strip=True),
            'games_missed':  cols[5].get_text(strip=True) if len(cols) > 5 else '0',
        })
    return injuries

# 50 top Premier League + La Liga players with their TM IDs
players = [
    (418560,  'erling-haaland'),
    (342229,  'kylian-mbappe'),
    (148455,  'mohamed-salah'),
    (352484,  'vinicius-junior'),
    (406635,  'jude-bellingham'),
    (433177,  'bukayo-saka'),
    (371998,  'phil-foden'),
    (580195,  'florian-wirtz'),
    (243045,  'harry-kane'),
    (357930,  'marcus-rashford'),
    (258923,  'bruno-fernandes'),
    (234793,  'kevin-de-bruyne'),
    (276669,  'trent-alexander-arnold'),
    (339364,  'rodri'),
    (357844,  'pedri'),
    (532692,  'gavi'),
    (581678,  'lamine-yamal'),
    (234591,  'robert-lewandowski'),
    (28003,   'lionel-messi'),
    (177683,  'cristiano-ronaldo'),
    (238223,  'neymar'),
    (401923,  'raphinha'),
    (154807,  'virgil-van-dijk'),
    (244723,  'alisson'),
    (234732,  'ederson'),
    (267798,  'thibaut-courtois'),
    (163469,  'luka-modric'),
    (68290,   'james-milner'),
    (170527,  'sadio-mane'),
    (339099,  'darwin-nunez'),
    (331845,  'gabriel-martinelli'),
    (315071,  'martin-odegaard'),
    (186920,  'heung-min-son'),
    (234566,  'pierre-emerick-aubameyang'),
    (120069,  'antoine-griezmann'),
    (166939,  'alvaro-morata'),
    (258556,  'marco-asensio'),
    (223232,  'ferran-torres'),
    (394560,  'ansu-fati'),
    (326031,  'ousmane-dembele'),
    (200512,  'ivan-rakitic'),
    (156859,  'sergio-busquets'),
    (341092,  'frenkie-de-jong'),
    (372613,  'pedri'),
    (259147,  'dani-olmo'),
    (348011,  'niclas-fullkrug'),
    (401333,  'jamal-musiala'),
    (580196,  'xavi-simons'),
    (357612,  'leroy-sane'),
    (234424,  'thomas-muller'),
]

all_injuries = []
for pid, pname in players:
    try:
        injuries = get_injuries(pid, pname)
        all_injuries.extend(injuries)
        print(f"{pname}: {len(injuries)} injury records")
        time.sleep(2)   # polite delay
    except Exception as e:
        print(f"{pname}: failed — {e}")

df_injuries = pd.DataFrame(all_injuries)
print(f"\nTotal injury records: {len(df_injuries)}")
print(df_injuries.head(10))

In [ ]:
# Clean days_missed column
def parse_days(val):
    try:
        return int(str(val).replace(' days','').replace('-','0').strip())
    except:
        return 0

df_injuries['days_missed_clean'] = df_injuries['days_missed'].apply(parse_days)

# Summary stats
print("=== Injury Dataset Summary ===")
print(f"Total records:    {len(df_injuries)}")
print(f"Unique players:   {df_injuries['player_name'].nunique()}")
print(f"Unique injuries:  {df_injuries['injury'].nunique()}")
print(f"\nTop injury types:")
print(df_injuries['injury'].value_counts().head(10))
print(f"\nAvg days missed: {df_injuries['days_missed_clean'].mean():.1f}")

# Save
df_injuries.to_csv(f'{folder}/injury_history.csv', index=False)
size = os.path.getsize(f'{folder}/injury_history.csv') / 1024
print(f"\nSaved injury_history.csv — {size:.1f} KB, {len(df_injuries)} rows")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os

folder = '/content/drive/MyDrive/TransferIQ Project'

# Load all saved datasets
df_players     = pd.read_csv(f'{folder}/players.csv')
df_valuations  = pd.read_csv(f'{folder}/market_valuations.csv')
df_transfers   = pd.read_csv(f'{folder}/transfer_history.csv')
df_appearances = pd.read_csv(f'{folder}/appearances.csv')
df_injuries    = pd.read_csv(f'{folder}/injury_history.csv')
df_sentiment   = pd.read_csv(f'{folder}/sentiment_data.csv')

print("Loaded successfully!")
for name, df in {
    'players':df_players,'valuations':df_valuations,
    'transfers':df_transfers,'appearances':df_appearances,
    'injuries':df_injuries,'sentiment':df_sentiment
}.items():
    print(f"  {name:<15} {df.shape[0]:>8,} rows  {df.shape[1]:>3} cols")


In [ ]:
# ── 2A: Missing value report ──────────────────────────────
def missing_report(df, name):
    m = df.isnull().sum()
    p = (m / len(df) * 100).round(2)
    r = pd.DataFrame({'missing': m, 'pct': p})
    r = r[r['missing'] > 0].sort_values('pct', ascending=False)
    print(f"\n{'='*40}\n{name}\n{'='*40}")
    print(r if len(r) > 0 else "  No missing values")

missing_report(df_players,     "Players")
missing_report(df_appearances, "Appearances")
missing_report(df_injuries,    "Injuries")
missing_report(df_transfers,   "Transfers")
missing_report(df_sentiment,   "Sentiment")

In [ ]:
# ── 2B: Fix missing values ────────────────────────────────

# Players
df_players['market_value_in_eur']         = df_players['market_value_in_eur'].fillna(0)
df_players['highest_market_value_in_eur'] = df_players['highest_market_value_in_eur'].fillna(0)
df_players['position']                    = df_players['position'].fillna('Unknown')
df_players['sub_position']                = df_players['sub_position'].fillna('Unknown')
df_players['foot']                        = df_players['foot'].fillna('Unknown')
df_players['country_of_citizenship']      = df_players['country_of_citizenship'].fillna('Unknown')
df_players['current_club_name']           = df_players['current_club_name'].fillna('Unknown')
df_players['height_in_cm']                = df_players['height_in_cm'].fillna(
                                                df_players['height_in_cm'].median())

# Appearances
for col in ['goals','assists','minutes_played','yellow_cards','red_cards']:
    if col in df_appearances.columns:
        df_appearances[col] = df_appearances[col].fillna(0)

# Injuries
df_injuries['days_missed']  = df_injuries['days_missed'].fillna(0)
df_injuries['games_missed'] = df_injuries['games_missed'].fillna(0)
df_injuries['injury']       = df_injuries['injury'].fillna('Unknown')

# Transfers
df_transfers['transfer_fee'] = df_transfers['transfer_fee'].fillna(0)

print("Missing values fixed!")

In [ ]:
# ── 2C: Fix data types ────────────────────────────────────
# Dates
df_players['date_of_birth']      = pd.to_datetime(df_players['date_of_birth'],      errors='coerce')
df_players['contract_expiry_date']= pd.to_datetime(df_players.get('contract_expiry_date'), errors='coerce')
df_transfers['transfer_date']    = pd.to_datetime(df_transfers['transfer_date'],    errors='coerce')
df_injuries['from_date']         = pd.to_datetime(df_injuries['from_date'],         errors='coerce')
df_injuries['until_date']        = pd.to_datetime(df_injuries.get('until_date'),    errors='coerce')

# Numerics
for col in ['goals','assists','minutes_played','yellow_cards','red_cards']:
    if col in df_appearances.columns:
        df_appearances[col] = pd.to_numeric(df_appearances[col], errors='coerce').fillna(0).astype(int)

df_injuries['days_missed']  = pd.to_numeric(df_injuries['days_missed'],  errors='coerce').fillna(0).astype(int)
df_injuries['games_missed'] = pd.to_numeric(df_injuries['games_missed'], errors='coerce').fillna(0).astype(int)

print("Data types fixed!")

In [ ]:
# ── 2D: Remove duplicates ─────────────────────────────────
before = {
    'players':     len(df_players),
    'appearances': len(df_appearances),
    'injuries':    len(df_injuries),
    'transfers':   len(df_transfers),
}

df_players     = df_players.drop_duplicates(subset='player_id', keep='first')
df_appearances = df_appearances.drop_duplicates()
df_injuries    = df_injuries.drop_duplicates()
df_transfers   = df_transfers.drop_duplicates()

print("Duplicates removed:")
for name, b in before.items():
    df = {'players':df_players,'appearances':df_appearances,
          'injuries':df_injuries,'transfers':df_transfers}[name]
    print(f"  {name:<15} {b:>8,} → {len(df):>8,}  (removed {b-len(df):,})")

In [ ]:
# ── 3A: Age & contract features ──────────────────────────
today = pd.Timestamp.today()

# Age
df_players['age'] = (
    (today - df_players['date_of_birth']).dt.days / 365.25
).round(1)

# Age group
def age_group(age):
    if age < 21:   return 'youngster'
    elif age < 25: return 'young'
    elif age < 29: return 'prime'
    elif age < 32: return 'experienced'
    else:          return 'veteran'

df_players['age_group'] = df_players['age'].apply(age_group)

# Contract remaining days
if 'contract_expiry_date' in df_players.columns:
    df_players['contract_days_remaining'] = (
        df_players['contract_expiry_date'] - today
    ).dt.days.clip(lower=0).fillna(0)

    df_players['contract_years_remaining'] = (
        df_players['contract_days_remaining'] / 365.25
    ).round(2)

    def contract_status(days):
        if days <= 0:     return 'expired'
        elif days < 180:  return 'expiring_soon'
        elif days < 365:  return 'one_year_left'
        elif days < 730:  return 'two_years_left'
        else:             return 'long_term'

    df_players['contract_status'] = df_players['contract_days_remaining'].apply(contract_status)
else:
    df_players['contract_days_remaining']  = 0
    df_players['contract_years_remaining'] = 0
    df_players['contract_status']          = 'unknown'

print("Age + Contract features:")
print(df_players[['name','age','age_group',
                   'contract_days_remaining',
                   'contract_status']].head(10))

In [ ]:
# ── 3A: Age & contract features ──────────────────────────
today = pd.Timestamp.today()

# Age
df_players['date_of_birth'] = pd.to_datetime(
    df_players['date_of_birth'], errors='coerce')

df_players['age'] = (
    (today - df_players['date_of_birth']).dt.days / 365.25
).round(1)

# Age group
def age_group(age):
    if pd.isna(age):   return 'unknown'
    elif age < 21:     return 'youngster'
    elif age < 25:     return 'young'
    elif age < 29:     return 'prime'
    elif age < 32:     return 'experienced'
    else:              return 'veteran'

df_players['age_group'] = df_players['age'].apply(age_group)

# Contract remaining days — safely handle missing column
if 'contract_expiry_date' in df_players.columns:
    # Force convert to datetime properly
    df_players['contract_expiry_date'] = pd.to_datetime(
        df_players['contract_expiry_date'], errors='coerce')

    # Now subtraction works
    df_players['contract_days_remaining'] = (
        df_players['contract_expiry_date'] - today
    ).dt.days.clip(lower=0).fillna(0).astype(int)

    df_players['contract_years_remaining'] = (
        df_players['contract_days_remaining'] / 365.25
    ).round(2)

    def contract_status(days):
        if days <= 0:     return 'expired'
        elif days < 180:  return 'expiring_soon'
        elif days < 365:  return 'one_year_left'
        elif days < 730:  return 'two_years_left'
        else:             return 'long_term'

    df_players['contract_status'] = \
        df_players['contract_days_remaining'].apply(contract_status)

else:
    # Column doesn't exist in this dataset — use defaults
    df_players['contract_days_remaining']  = 0
    df_players['contract_years_remaining'] = 0
    df_players['contract_status']          = 'unknown'
    print("Note: contract_expiry_date not found — using defaults")

# Verify output
print("Done!")
print(df_players[['name','age','age_group',
                   'contract_days_remaining',
                   'contract_status']].head(10))

In [ ]:
# ── 3B: Performance trend features ───────────────────────
perf = df_appearances.groupby('player_id').agg(
    total_goals          = ('goals',          'sum'),
    total_assists        = ('assists',        'sum'),
    total_appearances    = ('goals',          'count'),
    total_minutes        = ('minutes_played', 'sum'),
    avg_goals_per_game   = ('goals',          'mean'),
    avg_assists_per_game = ('assists',        'mean'),
    total_yellow_cards   = ('yellow_cards',   'sum'),
    total_red_cards      = ('red_cards',      'sum'),
).reset_index()

# Goal contributions
perf['goal_contributions'] = perf['total_goals'] + perf['total_assists']

# Goals per 90 minutes
perf['goals_per_90'] = np.where(
    perf['total_minutes'] > 0,
    (perf['total_goals'] / perf['total_minutes'] * 90).round(3),
    0
)

# Assists per 90 minutes
perf['assists_per_90'] = np.where(
    perf['total_minutes'] > 0,
    (perf['total_assists'] / perf['total_minutes'] * 90).round(3),
    0
)

# Discipline score (lower = better)
perf['discipline_score'] = (
    perf['total_yellow_cards'] * 1 +
    perf['total_red_cards']    * 3
)

# Performance score (composite)
perf['performance_score'] = (
    perf['goals_per_90']   * 40 +
    perf['assists_per_90'] * 25 +
    (perf['total_appearances'] / perf['total_appearances'].max() * 20) +
    (1 - perf['discipline_score'].clip(0,10) / 10) * 15
).round(2)

print("Performance features:")
print(perf[['player_id','total_goals','total_assists',
            'goals_per_90','assists_per_90',
            'performance_score']].head(10))

In [ ]:
# ── 3C: Injury risk features ─────────────────────────────
inj = df_injuries.groupby('player_id').agg(
    total_injuries   = ('injury',       'count'),
    total_days_out   = ('days_missed',  'sum'),
    total_games_out  = ('games_missed', 'sum'),
    avg_days_per_inj = ('days_missed',  'mean'),
    max_days_single  = ('days_missed',  'max'),
).reset_index()

# Injury risk score (0–100)
max_inj  = inj['total_injuries'].max()  or 1
max_days = inj['total_days_out'].max()  or 1

inj['injury_risk_score'] = (
    (inj['total_injuries'] / max_inj  * 50) +
    (inj['total_days_out'] / max_days * 50)
).round(2)

def risk_label(score):
    if score >= 70:   return 'high'
    elif score >= 40: return 'medium'
    else:             return 'low'

inj['injury_risk_label'] = inj['injury_risk_score'].apply(risk_label)

# Chronic injury flag (more than 4 injuries)
inj['chronic_injury_flag'] = (inj['total_injuries'] > 4).astype(int)

print("Injury risk features:")
print(inj[['player_id','total_injuries','total_days_out',
           'injury_risk_score','injury_risk_label',
           'chronic_injury_flag']].head(10))

In [ ]:
# ── 3D: Market value trend features ──────────────────────
df_valuations['date'] = pd.to_datetime(df_valuations['date'], errors='coerce')
df_valuations = df_valuations.sort_values(['player_id','date'])

val_trend = df_valuations.groupby('player_id').agg(
    first_value   = ('market_value_in_eur', 'first'),
    latest_value  = ('market_value_in_eur', 'last'),
    max_value     = ('market_value_in_eur', 'max'),
    avg_value     = ('market_value_in_eur', 'mean'),
    value_records = ('market_value_in_eur', 'count'),
).reset_index()

# Value growth (latest vs first)
val_trend['value_growth_pct'] = np.where(
    val_trend['first_value'] > 0,
    ((val_trend['latest_value'] - val_trend['first_value'])
     / val_trend['first_value'] * 100).round(2),
    0
)

# Value trend direction
val_trend['value_trend'] = val_trend['value_growth_pct'].apply(
    lambda x: 'rising' if x > 10 else ('falling' if x < -10 else 'stable'))

print("Market value trend features:")
print(val_trend[['player_id','first_value','latest_value',
                 'value_growth_pct','value_trend']].head(10))

In [ ]:
# ── 4A: One-hot encoding ─────────────────────────────────
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

# Position encoding
pos_dummies      = pd.get_dummies(df_players['position'],       prefix='pos')
foot_dummies     = pd.get_dummies(df_players['foot'],           prefix='foot')
contract_dummies = pd.get_dummies(df_players['contract_status'],prefix='contract')
agegroup_dummies = pd.get_dummies(df_players['age_group'],      prefix='age')

# Country — label encode (too many unique values)
le = LabelEncoder()
df_players['country_encoded'] = le.fit_transform(
    df_players['country_of_citizenship'].fillna('Unknown'))

print("One-hot encoding done!")
print("Position cols:",  pos_dummies.columns.tolist())
print("Foot cols:",      foot_dummies.columns.tolist())
print("Contract cols:",  contract_dummies.columns.tolist())
print("Age group cols:", agegroup_dummies.columns.tolist())

In [ ]:
# ── 4B: Scale numerical features ─────────────────────────
scaler = MinMaxScaler()

scale_cols = [c for c in [
    'age', 'height_in_cm',
    'market_value_in_eur',
    'highest_market_value_in_eur',
    'contract_days_remaining',
] if c in df_players.columns]

df_players_scaled = df_players.copy()
df_players_scaled[scale_cols] = scaler.fit_transform(
    df_players[scale_cols].fillna(0))

print("Scaling done!")
print(df_players_scaled[scale_cols].describe().round(3))

In [ ]:
# ── 5A: Run VADER on all text ─────────────────────────────
!pip install vaderSentiment --quiet
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

df_sentiment['compound'] = df_sentiment['text'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound'])
df_sentiment['positive'] = df_sentiment['text'].apply(
    lambda x: analyzer.polarity_scores(str(x))['pos'])
df_sentiment['negative'] = df_sentiment['text'].apply(
    lambda x: analyzer.polarity_scores(str(x))['neg'])
df_sentiment['neutral']  = df_sentiment['text'].apply(
    lambda x: analyzer.polarity_scores(str(x))['neu'])

df_sentiment['sentiment_label'] = df_sentiment['compound'].apply(
    lambda x: 'positive' if x >= 0.05 else
             ('negative' if x <= -0.05 else 'neutral'))

print("=== Preliminary Sentiment Analysis Report ===")
print(f"\nTotal headlines analysed : {len(df_sentiment)}")
print(f"Average compound score   : {df_sentiment['compound'].mean():.4f}")
print(f"Most positive score      : {df_sentiment['compound'].max():.4f}")
print(f"Most negative score      : {df_sentiment['compound'].min():.4f}")
print(f"\nSentiment breakdown:")
print(df_sentiment['sentiment_label'].value_counts())
print(f"\nBy source:")
print(df_sentiment.groupby('source')['compound'].mean().round(3))

In [ ]:
# ── 5B: Sentiment visualizations ─────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Sentiment Analysis Report — TransferIQ', fontsize=13)

# Distribution of compound scores
axes[0].hist(df_sentiment['compound'], bins=30,
             color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.7)
axes[0].set_title('Compound score distribution')
axes[0].set_xlabel('Compound score (-1 to +1)')

# Sentiment label pie chart
counts = df_sentiment['sentiment_label'].value_counts()
axes[1].pie(counts, labels=counts.index,
            colors=['#2ecc71','#e74c3c','#95a5a6'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Sentiment breakdown')

# Average by source
if 'source' in df_sentiment.columns:
    df_sentiment.groupby('source')['compound'].mean().plot(
        kind='bar', ax=axes[2], color='coral', rot=30)
    axes[2].set_title('Avg sentiment by source')
    axes[2].set_ylabel('Compound score')
    axes[2].axhline(0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(f'{folder}/sentiment_report.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sentiment report saved!")

In [ ]:
# Merge all features into one master dataframe
df_master = df_players[[
    'player_id','name','position','age','age_group',
    'market_value_in_eur','highest_market_value_in_eur',
    'foot','height_in_cm','current_club_name',
    'country_of_citizenship','country_encoded',
    'contract_days_remaining','contract_years_remaining',
    'contract_status'
]].copy()

# Add performance features
df_master = pd.merge(df_master, perf,     on='player_id', how='left')

# Add injury features
df_master = pd.merge(df_master, inj[[
    'player_id','total_injuries','total_days_out',
    'injury_risk_score','injury_risk_label','chronic_injury_flag'
]],                              on='player_id', how='left')

# Add market value trend
df_master = pd.merge(df_master, val_trend[[
    'player_id','value_growth_pct','value_trend','max_value'
]],                              on='player_id', how='left')

# Add one-hot encoded columns
df_master = pd.concat([
    df_master.reset_index(drop=True),
    pos_dummies.reset_index(drop=True),
    foot_dummies.reset_index(drop=True),
    contract_dummies.reset_index(drop=True),
    agegroup_dummies.reset_index(drop=True),
], axis=1)

# Fill any remaining nulls
df_master = df_master.fillna(0)

print(f"Master dataset shape: {df_master.shape}")
print(f"Total features: {df_master.shape[1]}")
print(df_master.head(3))

In [ ]:
# Save everything
df_master.to_csv(     f'{folder}/master_dataset.csv',      index=False)
df_sentiment.to_csv(  f'{folder}/sentiment_cleaned.csv',   index=False)
df_players.to_csv(    f'{folder}/players_cleaned.csv',     index=False)
df_appearances.to_csv(f'{folder}/appearances_cleaned.csv', index=False)
df_injuries.to_csv(   f'{folder}/injuries_cleaned.csv',    index=False)

print("\n=== MILESTONE 2 COMPLETE ===")
print(f"\n{'File':<35} {'Rows':>8} {'Cols':>6} {'Size':>10}")
print("-" * 63)
for fname in sorted(os.listdir(folder)):
    if fname.endswith('.csv'):
        path = f'{folder}/{fname}'
        df   = pd.read_csv(path, nrows=1)
        size = os.path.getsize(path) / 1024
        rows = sum(1 for _ in open(path)) - 1
        print(f"{fname:<35} {rows:>8,} {len(df.columns):>6} {size:>8.1f} KB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
import os

folder = '/content/drive/MyDrive/TransferIQ Project'

df_players     = pd.read_csv(f'{folder}/players_cleaned.csv')
df_appearances = pd.read_csv(f'{folder}/appearances_cleaned.csv')
df_injuries    = pd.read_csv(f'{folder}/injuries_cleaned.csv')
df_valuations  = pd.read_csv(f'{folder}/market_valuations.csv')
df_sentiment   = pd.read_csv(f'{folder}/sentiment_cleaned.csv')
df_master      = pd.read_csv(f'{folder}/master_dataset.csv')

# Fix date columns after CSV reload
df_players['date_of_birth']      = pd.to_datetime(df_players['date_of_birth'],      errors='coerce')
df_valuations['date']            = pd.to_datetime(df_valuations['date'],            errors='coerce')
df_injuries['from_date']         = pd.to_datetime(df_injuries['from_date'],         errors='coerce')
df_appearances['season']         = df_appearances['season'].astype(str)

today = pd.Timestamp.today()

print("All datasets loaded!")
for name, df in {
    'players':df_players, 'appearances':df_appearances,
    'injuries':df_injuries, 'valuations':df_valuations,
    'sentiment':df_sentiment, 'master':df_master
}.items():
    print(f"  {name:<15} {df.shape[0]:>8,} rows  {df.shape[1]:>3} cols")

In [ ]:
# ── Check what columns actually exist ────────────────────
print("Appearances columns:")
print(df_appearances.columns.tolist())
print("\nSample data:")
print(df_appearances.head(3))

In [ ]:
def date_to_season(date):
    if pd.isna(date):
        return 'Unknown'
    year  = date.year
    month = date.month
    if month >= 7:
        return f"{year}-{str(year+1)[-2:]}"
    else:
        return f"{year-1}-{str(year)[-2:]}"

df_appearances['season'] = df_appearances['date'].apply(date_to_season)

print("Season column created!")
print(df_appearances[['date','season','player_name',
                       'goals','assists','minutes_played']].head(8))
print(f"\nUnique seasons: {len(df_appearances['season'].unique())}")
print(sorted(df_appearances['season'].unique())[:10])

In [ ]:
df_players     = pd.read_csv(f'{folder}/players_cleaned.csv')
df_appearances = pd.read_csv(f'{folder}/appearances_cleaned.csv')
df_injuries    = pd.read_csv(f'{folder}/injuries_cleaned.csv')
df_valuations  = pd.read_csv(f'{folder}/market_valuations.csv')
df_sentiment   = pd.read_csv(f'{folder}/sentiment_cleaned.csv')

# Fix date columns after CSV reload
df_players['date_of_birth'] = pd.to_datetime(df_players['date_of_birth'],   errors='coerce')
df_appearances['date']      = pd.to_datetime(df_appearances['date'],         errors='coerce')
df_injuries['from_date']    = pd.to_datetime(df_injuries.get('from_date'),   errors='coerce')
df_valuations['date']       = pd.to_datetime(df_valuations['date'],          errors='coerce')

today = pd.Timestamp.today()

print("Datasets loaded!")
for name, df in {
    'players':df_players,       'appearances':df_appearances,
    'injuries':df_injuries,     'valuations':df_valuations,
    'sentiment':df_sentiment
}.items():
    print(f"  {name:<15} {df.shape[0]:>8,} rows  {df.shape[1]:>3} cols")

In [ ]:
def date_to_season(date):
    if pd.isna(date):
        return 'Unknown'
    year  = date.year
    month = date.month
    if month >= 7:
        return f"{year}-{str(year+1)[-2:]}"
    else:
        return f"{year-1}-{str(year)[-2:]}"

df_appearances['season'] = df_appearances['date'].apply(date_to_season)

print("Season column created!")
print(df_appearances[['date','season','player_name',
                       'goals','assists','minutes_played']].head(8))
print(f"\nUnique seasons: {len(df_appearances['season'].unique())}")
print(sorted(df_appearances['season'].unique())[:10])

In [ ]:
for col in ['goals','assists','minutes_played','yellow_cards','red_cards']:
    df_appearances[col] = pd.to_numeric(
        df_appearances[col], errors='coerce').fillna(0).astype(int)

df_injuries['days_missed']  = pd.to_numeric(
    df_injuries['days_missed'],  errors='coerce').fillna(0).astype(int)
df_injuries['games_missed'] = pd.to_numeric(
    df_injuries['games_missed'], errors='coerce').fillna(0).astype(int)

print("Numeric columns fixed!")
print(df_appearances.dtypes)

In [ ]:
df_appearances = df_appearances.sort_values(['player_id','date'])

season_stats = df_appearances.groupby(['player_id','season']).agg(
    season_goals        = ('goals',          'sum'),
    season_assists      = ('assists',        'sum'),
    season_minutes      = ('minutes_played', 'sum'),
    season_apps         = ('goals',          'count'),
    season_yellow_cards = ('yellow_cards',   'sum'),
    season_red_cards    = ('red_cards',      'sum'),
).reset_index()

print(f"Season stats: {season_stats.shape}")
print(season_stats.head(8))

In [ ]:
# Trend slope function
def calc_slope(values):
    if len(values) < 2:
        return 0.0
    x = np.arange(len(values))
    slope, _, _, _, _ = stats.linregress(x, values)
    return round(slope, 4)

# Goals trend slope per player
goals_trend = season_stats.groupby('player_id')['season_goals'].apply(
    lambda x: calc_slope(x.values)).reset_index()
goals_trend.columns = ['player_id','goals_trend_slope']

# Assists trend slope
assists_trend = season_stats.groupby('player_id')['season_assists'].apply(
    lambda x: calc_slope(x.values)).reset_index()
assists_trend.columns = ['player_id','assists_trend_slope']

# Minutes trend (playing time)
minutes_trend = season_stats.groupby('player_id')['season_minutes'].apply(
    lambda x: calc_slope(x.values)).reset_index()
minutes_trend.columns = ['player_id','minutes_trend_slope']

# Peak and average per player
peak_stats = season_stats.groupby('player_id').agg(
    peak_season_goals   = ('season_goals',        'max'),
    peak_season_assists = ('season_assists',      'max'),
    seasons_played      = ('season',              'count'),
    avg_season_goals    = ('season_goals',        'mean'),
    avg_season_assists  = ('season_assists',      'mean'),
    avg_season_minutes  = ('season_minutes',      'mean'),
    total_yellow_cards  = ('season_yellow_cards', 'sum'),
    total_red_cards     = ('season_red_cards',    'sum'),
).reset_index()

# Merge trend features
perf_trend = peak_stats \
    .merge(goals_trend,   on='player_id', how='left') \
    .merge(assists_trend, on='player_id', how='left') \
    .merge(minutes_trend, on='player_id', how='left')

# Trend label
def trend_label(slope):
    if slope >  0.5:   return 'rising_fast'
    elif slope > 0:    return 'rising'
    elif slope > -0.5: return 'declining'
    else:              return 'declining_fast'

perf_trend['performance_trend']  = perf_trend['goals_trend_slope'].apply(trend_label)
perf_trend['discipline_score']   = (
    perf_trend['total_yellow_cards'] * 1 +
    perf_trend['total_red_cards']    * 3)

print("Performance trend features done!")
print(perf_trend[['player_id','avg_season_goals','goals_trend_slope',
                   'seasons_played','performance_trend']].head(8))

In [ ]:
# Trend slope function
def calc_slope(values):
    if len(values) < 2:
        return 0.0
    x = np.arange(len(values))
    slope, _, _, _, _ = stats.linregress(x, values)
    return round(slope, 4)

# Goals trend slope per player
goals_trend = season_stats.groupby('player_id')['season_goals'].apply(
    lambda x: calc_slope(x.values)).reset_index()
goals_trend.columns = ['player_id','goals_trend_slope']

# Assists trend slope
assists_trend = season_stats.groupby('player_id')['season_assists'].apply(
    lambda x: calc_slope(x.values)).reset_index()
assists_trend.columns = ['player_id','assists_trend_slope']

# Minutes trend (playing time)
minutes_trend = season_stats.groupby('player_id')['season_minutes'].apply(
    lambda x: calc_slope(x.values)).reset_index()
minutes_trend.columns = ['player_id','minutes_trend_slope']

# Peak and average per player
peak_stats = season_stats.groupby('player_id').agg(
    peak_season_goals   = ('season_goals',        'max'),
    peak_season_assists = ('season_assists',      'max'),
    seasons_played      = ('season',              'count'),
    avg_season_goals    = ('season_goals',        'mean'),
    avg_season_assists  = ('season_assists',      'mean'),
    avg_season_minutes  = ('season_minutes',      'mean'),
    total_yellow_cards  = ('season_yellow_cards', 'sum'),
    total_red_cards     = ('season_red_cards',    'sum'),
).reset_index()

# Merge trend features
perf_trend = peak_stats \
    .merge(goals_trend,   on='player_id', how='left') \
    .merge(assists_trend, on='player_id', how='left') \
    .merge(minutes_trend, on='player_id', how='left')

# Trend label
def trend_label(slope):
    if slope >  0.5:   return 'rising_fast'
    elif slope > 0:    return 'rising'
    elif slope > -0.5: return 'declining'
    else:              return 'declining_fast'

perf_trend['performance_trend']  = perf_trend['goals_trend_slope'].apply(trend_label)
perf_trend['discipline_score']   = (
    perf_trend['total_yellow_cards'] * 1 +
    perf_trend['total_red_cards']    * 3)

print("Performance trend features done!")
print(perf_trend[['player_id','avg_season_goals','goals_trend_slope',
                   'seasons_played','performance_trend']].head(8))

In [ ]:
from scipy import stats
import numpy as np
print("stats imported!")

In [ ]:
# Trend slope function
def calc_slope(values):
    if len(values) < 2:
        return 0.0
    x = np.arange(len(values))
    slope, _, _, _, _ = stats.linregress(x, values)
    return round(slope, 4)

# Goals trend slope per player
goals_trend = season_stats.groupby('player_id')['season_goals'].apply(
    lambda x: calc_slope(x.values)).reset_index()
goals_trend.columns = ['player_id','goals_trend_slope']

# Assists trend slope
assists_trend = season_stats.groupby('player_id')['season_assists'].apply(
    lambda x: calc_slope(x.values)).reset_index()
assists_trend.columns = ['player_id','assists_trend_slope']

# Minutes trend (playing time)
minutes_trend = season_stats.groupby('player_id')['season_minutes'].apply(
    lambda x: calc_slope(x.values)).reset_index()
minutes_trend.columns = ['player_id','minutes_trend_slope']

# Peak and average per player
peak_stats = season_stats.groupby('player_id').agg(
    peak_season_goals   = ('season_goals',        'max'),
    peak_season_assists = ('season_assists',      'max'),
    seasons_played      = ('season',              'count'),
    avg_season_goals    = ('season_goals',        'mean'),
    avg_season_assists  = ('season_assists',      'mean'),
    avg_season_minutes  = ('season_minutes',      'mean'),
    total_yellow_cards  = ('season_yellow_cards', 'sum'),
    total_red_cards     = ('season_red_cards',    'sum'),
).reset_index()

# Merge trend features
perf_trend = peak_stats \
    .merge(goals_trend,   on='player_id', how='left') \
    .merge(assists_trend, on='player_id', how='left') \
    .merge(minutes_trend, on='player_id', how='left')

# Trend label
def trend_label(slope):
    if slope >  0.5:   return 'rising_fast'
    elif slope > 0:    return 'rising'
    elif slope > -0.5: return 'declining'
    else:              return 'declining_fast'

perf_trend['performance_trend']  = perf_trend['goals_trend_slope'].apply(trend_label)
perf_trend['discipline_score']   = (
    perf_trend['total_yellow_cards'] * 1 +
    perf_trend['total_red_cards']    * 3)

print("Performance trend features done!")
print(perf_trend[['player_id','avg_season_goals','goals_trend_slope',
                   'seasons_played','performance_trend']].head(8))

In [ ]:
# Consistency score
consistency = season_stats.groupby('player_id')['season_goals'].agg(
    ['mean','std']).reset_index()
consistency.columns = ['player_id','mean_goals','std_goals']
consistency['std_goals']        = consistency['std_goals'].fillna(0)
consistency['consistency_score']= (
    1 - (consistency['std_goals'] / (consistency['mean_goals'] + 1e-5))
).clip(0,1).round(4)

# Form score (recent 2 seasons vs career average)
def form_score(player_df):
    if len(player_df) < 3:
        return 0.0
    recent = player_df['season_goals'].iloc[-2:].mean()
    career = player_df['season_goals'].mean()
    return round((recent - career) / (career + 1e-5), 4)

form = season_stats.groupby('player_id').apply(form_score).reset_index()
form.columns = ['player_id','form_score']
form['form_label'] = form['form_score'].apply(
    lambda x: 'in_form'     if x >  0.1 else
             ('out_of_form' if x < -0.1 else 'average'))

print("Consistency & form done!")
print(consistency[['player_id','consistency_score']].head(5))
print(form[['player_id','form_score','form_label']].head(5))

In [ ]:
career = df_appearances.groupby('player_id').agg(
    career_goals   = ('goals',          'sum'),
    career_assists = ('assists',        'sum'),
    career_minutes = ('minutes_played', 'sum'),
    career_apps    = ('goals',          'count'),
).reset_index()

career['goals_per_90'] = np.where(
    career['career_minutes'] > 0,
    (career['career_goals'] / career['career_minutes'] * 90).round(3), 0)

career['assists_per_90'] = np.where(
    career['career_minutes'] > 0,
    (career['career_assists'] / career['career_minutes'] * 90).round(3), 0)

career['goal_contributions'] = career['career_goals'] + career['career_assists']

print("Career totals done!")
print(career[['player_id','career_goals','career_assists',
              'goals_per_90','assists_per_90']].head(8))

In [ ]:
# Injury severity mapping
severity_map = {
    'muscle':     3, 'knee':       5, 'ankle':   3,
    'hamstring':  4, 'back':       4, 'cruciate':9,
    'broken':     8, 'shoulder':   3, 'thigh':   3,
    'groin':      3, 'hip':        4, 'foot':    4,
    'calf':       2, 'illness':    1, 'suspension':1,
}

def get_severity(injury_text):
    text = str(injury_text).lower()
    for key, val in severity_map.items():
        if key in text:
            return val
    return 2

df_injuries['injury_severity'] = df_injuries['injury'].apply(get_severity)

# Injury features per player
inj_features = df_injuries.groupby('player_id').agg(
    total_injuries       = ('injury',           'count'),
    total_days_out       = ('days_missed',       'sum'),
    total_games_out      = ('games_missed',      'sum'),
    avg_days_per_injury  = ('days_missed',       'mean'),
    max_days_single_inj  = ('days_missed',       'max'),
    avg_severity         = ('injury_severity',   'mean'),
    max_severity         = ('injury_severity',   'max'),
    severe_injuries      = ('injury_severity',   lambda x: (x >= 7).sum()),
).reset_index()

# Injury risk score (0-100)
max_days = inj_features['total_days_out'].max() or 1
max_inj  = inj_features['total_injuries'].max()  or 1

inj_features['injury_risk_score'] = (
    (inj_features['total_days_out'] / max_days * 40) +
    (inj_features['avg_severity']   / 10       * 35) +
    (inj_features['total_injuries'] / max_inj  * 15) +
    (inj_features['severe_injuries']* 2              ).clip(0,10)
).round(2)

inj_features['injury_risk_label']   = inj_features['injury_risk_score'].apply(
    lambda x: 'high' if x >= 60 else ('medium' if x >= 30 else 'low'))
inj_features['chronic_injury_flag'] = (inj_features['total_injuries'] > 4).astype(int)

print("Injury features done!")
print(inj_features[['player_id','total_injuries','total_days_out',
                     'avg_severity','injury_risk_score',
                     'injury_risk_label']].head(8))

In [ ]:
df_valuations = df_valuations.sort_values(['player_id','date'])

val_trend = df_valuations.groupby('player_id').agg(
    first_value    = ('market_value_in_eur', 'first'),
    latest_value   = ('market_value_in_eur', 'last'),
    max_value      = ('market_value_in_eur', 'max'),
    avg_value      = ('market_value_in_eur', 'mean'),
    value_records  = ('market_value_in_eur', 'count'),
).reset_index()

val_trend['value_growth_pct'] = np.where(
    val_trend['first_value'] > 0,
    ((val_trend['latest_value'] - val_trend['first_value'])
     / val_trend['first_value'] * 100).round(2), 0)

val_trend['value_trend'] = val_trend['value_growth_pct'].apply(
    lambda x: 'rising' if x > 10 else ('falling' if x < -10 else 'stable'))

print("Market value trends done!")
print(val_trend[['player_id','first_value','latest_value',
                 'value_growth_pct','value_trend']].head(8))

In [ ]:
analyzer = SentimentIntensityAnalyzer()

# Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+',  '', text)
    text = re.sub(r'@\w+',     '', text)
    text = re.sub(r'#\w+',     '', text)
    text = re.sub(r'[^\w\s]',  '', text)
    text = re.sub(r'\s+',      ' ', text)
    return text.strip()

df_sentiment['text_clean'] = df_sentiment['text'].apply(clean_text)

# VADER scores
def get_vader(text):
    s = analyzer.polarity_scores(str(text))
    return pd.Series({
        'vader_compound': s['compound'],
        'vader_positive': s['pos'],
        'vader_negative': s['neg'],
        'vader_neutral':  s['neu'],
    })

vader_df     = df_sentiment['text_clean'].apply(get_vader)
df_sentiment = pd.concat([df_sentiment, vader_df], axis=1)

# TextBlob scores
def get_textblob(text):
    b = TextBlob(str(text))
    return pd.Series({
        'tb_polarity':     round(b.sentiment.polarity,     4),
        'tb_subjectivity': round(b.sentiment.subjectivity, 4),
    })

tb_df        = df_sentiment['text_clean'].apply(get_textblob)
df_sentiment = pd.concat([df_sentiment, tb_df], axis=1)

# Transfer keyword scoring
positive_kw = ['sign','deal','agree','brilliant','world class',
               'record','winner','champion','star','best']
negative_kw = ['injury','injured','crisis','miss','failed',
               'flop','disappoint','ban','suspend','struggle']

def keyword_score(text):
    text = str(text).lower()
    pos  = sum(1 for w in positive_kw if w in text)
    neg  = sum(1 for w in negative_kw if w in text)
    return round((pos - neg) / (pos + neg + 1e-5), 4)

df_sentiment['keyword_score'] = df_sentiment['text_clean'].apply(keyword_score)

# Combined market impact score
df_sentiment['combined_sentiment'] = (
    (df_sentiment['vader_compound'] + df_sentiment['tb_polarity']) / 2
).round(4)

df_sentiment['market_impact_score'] = (
    df_sentiment['vader_compound'] * 0.4 +
    df_sentiment['tb_polarity']    * 0.3 +
    df_sentiment['keyword_score']  * 0.3
).round(4)

df_sentiment['sentiment_label'] = df_sentiment['combined_sentiment'].apply(
    lambda x: 'positive' if x >= 0.05 else
             ('negative' if x <= -0.05 else 'neutral'))

df_sentiment['subjectivity_label'] = df_sentiment['tb_subjectivity'].apply(
    lambda x: 'opinion' if x > 0.5 else 'factual')

print("=== Sentiment Analysis Complete ===")
print(f"Total records:         {len(df_sentiment)}")
print(f"Avg VADER compound:    {df_sentiment['vader_compound'].mean():.4f}")
print(f"Avg TextBlob polarity: {df_sentiment['tb_polarity'].mean():.4f}")
print(f"\nSentiment breakdown:")
print(df_sentiment['sentiment_label'].value_counts())
print(f"\nSubjectivity:")
print(df_sentiment['subjectivity_label'].value_counts())

In [ ]:
# ── Run this whenever you get a NameError ────────────────
!pip install vaderSentiment textblob scipy --quiet

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os, re, warnings
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

folder = '/content/drive/MyDrive/TransferIQ Project'
today  = pd.Timestamp.today()

print("All imports ready!")

In [ ]:
analyzer = SentimentIntensityAnalyzer()

# Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+',  '', text)
    text = re.sub(r'@\w+',     '', text)
    text = re.sub(r'#\w+',     '', text)
    text = re.sub(r'[^\w\s]',  '', text)
    text = re.sub(r'\s+',      ' ', text)
    return text.strip()

df_sentiment['text_clean'] = df_sentiment['text'].apply(clean_text)

# VADER scores
def get_vader(text):
    s = analyzer.polarity_scores(str(text))
    return pd.Series({
        'vader_compound': s['compound'],
        'vader_positive': s['pos'],
        'vader_negative': s['neg'],
        'vader_neutral':  s['neu'],
    })

vader_df     = df_sentiment['text_clean'].apply(get_vader)
df_sentiment = pd.concat([df_sentiment, vader_df], axis=1)

# TextBlob scores
def get_textblob(text):
    b = TextBlob(str(text))
    return pd.Series({
        'tb_polarity':     round(b.sentiment.polarity,     4),
        'tb_subjectivity': round(b.sentiment.subjectivity, 4),
    })

tb_df        = df_sentiment['text_clean'].apply(get_textblob)
df_sentiment = pd.concat([df_sentiment, tb_df], axis=1)

# Transfer keyword scoring
positive_kw = ['sign','deal','agree','brilliant','world class',
               'record','winner','champion','star','best']
negative_kw = ['injury','injured','crisis','miss','failed',
               'flop','disappoint','ban','suspend','struggle']

def keyword_score(text):
    text = str(text).lower()
    pos  = sum(1 for w in positive_kw if w in text)
    neg  = sum(1 for w in negative_kw if w in text)
    return round((pos - neg) / (pos + neg + 1e-5), 4)

df_sentiment['keyword_score'] = df_sentiment['text_clean'].apply(keyword_score)

# Combined market impact score
df_sentiment['combined_sentiment'] = (
    (df_sentiment['vader_compound'] + df_sentiment['tb_polarity']) / 2
).round(4)

df_sentiment['market_impact_score'] = (
    df_sentiment['vader_compound'] * 0.4 +
    df_sentiment['tb_polarity']    * 0.3 +
    df_sentiment['keyword_score']  * 0.3
).round(4)

df_sentiment['sentiment_label'] = df_sentiment['combined_sentiment'].apply(
    lambda x: 'positive' if x >= 0.05 else
             ('negative' if x <= -0.05 else 'neutral'))

df_sentiment['subjectivity_label'] = df_sentiment['tb_subjectivity'].apply(
    lambda x: 'opinion' if x > 0.5 else 'factual')

print("=== Sentiment Analysis Complete ===")
print(f"Total records:         {len(df_sentiment)}")
print(f"Avg VADER compound:    {df_sentiment['vader_compound'].mean():.4f}")
print(f"Avg TextBlob polarity: {df_sentiment['tb_polarity'].mean():.4f}")
print(f"\nSentiment breakdown:")
print(df_sentiment['sentiment_label'].value_counts())
print(f"\nSubjectivity:")
print(df_sentiment['subjectivity_label'].value_counts())

In [ ]:
fig = plt.figure(figsize=(16,12))
fig.suptitle('TransferIQ — Milestone 3 Sentiment Analysis Report', fontsize=15)
gs  = gridspec.GridSpec(3, 3, figure=fig)

# 1. VADER compound distribution
ax1 = fig.add_subplot(gs[0,0])
ax1.hist(df_sentiment['vader_compound'], bins=30, color='steelblue', edgecolor='white')
ax1.axvline(0, color='red', linestyle='--', alpha=0.6)
ax1.set_title('VADER compound scores')
ax1.set_xlabel('Score (-1 to +1)')

# 2. TextBlob polarity
ax2 = fig.add_subplot(gs[0,1])
ax2.hist(df_sentiment['tb_polarity'], bins=30, color='seagreen', edgecolor='white')
ax2.axvline(0, color='red', linestyle='--', alpha=0.6)
ax2.set_title('TextBlob polarity')
ax2.set_xlabel('Score (-1 to +1)')

# 3. Subjectivity
ax3 = fig.add_subplot(gs[0,2])
ax3.hist(df_sentiment['tb_subjectivity'], bins=30, color='coral', edgecolor='white')
ax3.axvline(0.5, color='red', linestyle='--', alpha=0.6)
ax3.set_title('Subjectivity distribution')
ax3.set_xlabel('0=factual  1=opinion')

# 4. Sentiment label pie
ax4 = fig.add_subplot(gs[1,0])
counts = df_sentiment['sentiment_label'].value_counts()
ax4.pie(counts, labels=counts.index,
        colors=['#2ecc71','#e74c3c','#95a5a6'],
        autopct='%1.1f%%', startangle=90)
ax4.set_title('Sentiment breakdown')

# 5. Market impact score
ax5 = fig.add_subplot(gs[1,1])
ax5.hist(df_sentiment['market_impact_score'], bins=30,
         color='purple', edgecolor='white')
ax5.axvline(0, color='red', linestyle='--', alpha=0.6)
ax5.set_title('Market impact scores')

# 6. VADER vs TextBlob scatter
ax6 = fig.add_subplot(gs[1,2])
ax6.scatter(df_sentiment['vader_compound'],
            df_sentiment['tb_polarity'],
            alpha=0.4, color='teal', s=20)
ax6.axhline(0, color='gray', linestyle='--', alpha=0.4)
ax6.axvline(0, color='gray', linestyle='--', alpha=0.4)
ax6.set_title('VADER vs TextBlob')
ax6.set_xlabel('VADER compound')
ax6.set_ylabel('TextBlob polarity')

# 7. Sentiment by source
ax7 = fig.add_subplot(gs[2,:])
if 'source' in df_sentiment.columns:
    source_avg = df_sentiment.groupby('source')[[
        'vader_compound','tb_polarity','market_impact_score'
    ]].mean()
    source_avg.plot(kind='bar', ax=ax7, rot=30, colormap='viridis')
    ax7.axhline(0, color='black', linestyle='--', alpha=0.4)
    ax7.set_title('Average sentiment scores by source')
    ax7.set_ylabel('Score')
    ax7.legend(loc='upper right')

plt.tight_layout()
plt.savefig(f'{folder}/milestone3_sentiment_report.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sentiment report saved!")

In [ ]:
# Start with core player info
df_final = df_players[[
    'player_id','name','position','age',
    'market_value_in_eur','highest_market_value_in_eur',
    'foot','height_in_cm','current_club_name',
    'country_of_citizenship','contract_days_remaining',
    'contract_years_remaining','contract_status'
] if 'contract_days_remaining' in df_players.columns else [
    'player_id','name','position',
    'market_value_in_eur','highest_market_value_in_eur',
    'foot','height_in_cm','current_club_name',
    'country_of_citizenship'
]].copy()

# Add age if missing
if 'age' not in df_final.columns:
    df_players['date_of_birth'] = pd.to_datetime(
        df_players['date_of_birth'], errors='coerce')
    df_final['age'] = (
        (today - df_players['date_of_birth']).dt.days / 365.25
    ).round(1)

# Merge all feature tables
df_final = df_final \
    .merge(perf_trend[[
        'player_id','avg_season_goals','avg_season_assists',
        'peak_season_goals','seasons_played',
        'goals_trend_slope','assists_trend_slope',
        'minutes_trend_slope','performance_trend','discipline_score'
    ]], on='player_id', how='left') \
    .merge(consistency[['player_id','consistency_score']],
           on='player_id', how='left') \
    .merge(form[['player_id','form_score','form_label']],
           on='player_id', how='left') \
    .merge(career[['player_id','career_goals','career_assists',
                   'career_apps','goals_per_90',
                   'assists_per_90','goal_contributions']],
           on='player_id', how='left') \
    .merge(inj_features[[
        'player_id','total_injuries','total_days_out',
        'avg_severity','severe_injuries',
        'injury_risk_score','injury_risk_label','chronic_injury_flag'
    ]], on='player_id', how='left') \
    .merge(val_trend[[
        'player_id','max_value','avg_value',
        'value_growth_pct','value_trend'
    ]], on='player_id', how='left')

# Add global sentiment scores
df_final['avg_sentiment']     = round(df_sentiment['combined_sentiment'].mean(),    4)
df_final['avg_market_impact'] = round(df_sentiment['market_impact_score'].mean(),   4)
df_final['pct_positive_news'] = round(
    (df_sentiment['sentiment_label']=='positive').mean() * 100, 2)

# One-hot encode categoricals
pos_dummies      = pd.get_dummies(df_final['position'],        prefix='pos')
foot_dummies     = pd.get_dummies(df_final['foot'],            prefix='foot')
trend_dummies    = pd.get_dummies(df_final['performance_trend'],prefix='trend')
form_dummies     = pd.get_dummies(df_final['form_label'],      prefix='form')
risk_dummies     = pd.get_dummies(df_final['injury_risk_label'],prefix='risk')

df_final = pd.concat([
    df_final.reset_index(drop=True),
    pos_dummies.reset_index(drop=True),
    foot_dummies.reset_index(drop=True),
    trend_dummies.reset_index(drop=True),
    form_dummies.reset_index(drop=True),
    risk_dummies.reset_index(drop=True),
], axis=1)

# Fill remaining nulls
df_final = df_final.fillna(0)

print(f"Final feature set shape: {df_final.shape}")
print(f"Total features: {df_final.shape[1]}")

In [ ]:
df_final.to_csv(    f'{folder}/final_feature_set.csv',         index=False)
df_sentiment.to_csv(f'{folder}/sentiment_advanced.csv',        index=False)
perf_trend.to_csv(  f'{folder}/performance_features.csv',      index=False)
inj_features.to_csv(f'{folder}/injury_features.csv',           index=False)
val_trend.to_csv(   f'{folder}/valuation_trends.csv',          index=False)

print("=== MILESTONE 3 COMPLETE ===\n")
print(f"{'File':<38} {'Rows':>8} {'Size':>10}")
print("-" * 58)
for fname in sorted(os.listdir(folder)):
    if fname.endswith(('.csv','.png')):
        path = f'{folder}/{fname}'
        size = os.path.getsize(path) / 1024
        rows = sum(1 for _ in open(path)) - 1
        print(f"{fname:<38} {rows:>8,} {size:>8.1f} KB")

In [ ]:
# ── Cell 14 — Save & verify all deliverables ─────────────
df_final.to_csv(    f'{folder}/final_feature_set.csv',    index=False)
df_sentiment.to_csv(f'{folder}/sentiment_advanced.csv',   index=False)
perf_trend.to_csv(  f'{folder}/performance_features.csv', index=False)
inj_features.to_csv(f'{folder}/injury_features.csv',      index=False)
val_trend.to_csv(   f'{folder}/valuation_trends.csv',     index=False)

print("=== MILESTONE 3 COMPLETE ===\n")
print(f"{'File':<38} {'Rows':>8} {'Size':>10}")
print("-" * 58)

for fname in sorted(os.listdir(folder)):
    path = f'{folder}/{fname}'
    size = os.path.getsize(path) / 1024

    if fname.endswith('.png'):
        # PNG files — don't count rows
        print(f"{fname:<38} {'[image]':>8} {size:>8.1f} KB")

    elif fname.endswith('.csv'):
        # CSV files — safely count rows
        try:
            rows = sum(1 for _ in open(path, encoding='utf-8', errors='ignore')) - 1
            print(f"{fname:<38} {rows:>8,} {size:>8.1f} KB")
        except Exception as e:
            print(f"{fname:<38} {'[error]':>8} {size:>8.1f} KB")

In [ ]:
!pip install tensorflow scikit-learn matplotlib --quiet

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (LSTM, Dense, Dropout,
    RepeatVector, TimeDistributed, Input, Bidirectional)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

folder  = '/content/drive/MyDrive/TransferIQ Project'
models  = f'{folder}/models'
os.makedirs(models, exist_ok=True)

print(f"TensorFlow version: {tf.__version__}")
print("Ready!")

In [ ]:
df_final    = pd.read_csv(f'{folder}/final_feature_set.csv')
df_val      = pd.read_csv(f'{folder}/market_valuations.csv')
df_val['date'] = pd.to_datetime(df_val['date'], errors='coerce')
df_val      = df_val.sort_values(['player_id','date'])

print(f"Final feature set: {df_final.shape}")
print(f"Valuations:        {df_val.shape}")
print(f"\nValuation columns: {df_val.columns.tolist()}")
print(df_val.head(5))

In [ ]:
# ── Keep only players with enough valuation history ──────
player_counts = df_val.groupby('player_id')['market_value_in_eur'].count()
eligible      = player_counts[player_counts >= 8].index
df_val_filtered = df_val[df_val['player_id'].isin(eligible)].copy()

print(f"Eligible players (>=8 records): {len(eligible):,}")
print(f"Filtered valuations:            {len(df_val_filtered):,}")

# ── Scale market values ───────────────────────────────────
value_scaler = MinMaxScaler()
df_val_filtered['value_scaled'] = value_scaler.fit_transform(
    df_val_filtered[['market_value_in_eur']])

# ── Create sequences function ─────────────────────────────
def create_sequences(values, seq_len=6, horizon=1):
    """
    values   : array of scaled values for one player
    seq_len  : number of past time steps to look at
    horizon  : number of future steps to predict
    """
    X, y = [], []
    for i in range(len(values) - seq_len - horizon + 1):
        X.append(values[i : i + seq_len])
        y.append(values[i + seq_len : i + seq_len + horizon])
    return np.array(X), np.array(y)

SEQ_LEN = 6   # look back 6 time steps
HORIZON = 1   # predict 1 step ahead (univariate)

X_uni, y_uni = [], []

for pid in eligible:
    vals = df_val_filtered[
        df_val_filtered['player_id'] == pid
    ]['value_scaled'].values

    X_p, y_p = create_sequences(vals, SEQ_LEN, HORIZON)
    if len(X_p) > 0:
        X_uni.append(X_p)
        y_uni.append(y_p)

X_uni = np.vstack(X_uni)
y_uni = np.vstack(y_uni).squeeze()

print(f"\nUnivariate sequences:")
print(f"  X shape: {X_uni.shape}  (samples, timesteps)")
print(f"  y shape: {y_uni.shape}  (samples,)")

# Reshape for LSTM (samples, timesteps, features)
X_uni = X_uni.reshape(X_uni.shape[0], X_uni.shape[1], 1)

# Train / val / test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X_uni, y_uni, test_size=0.2, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr,  y_tr,  test_size=0.1, random_state=42)

print(f"\nTrain:      {X_tr.shape[0]:,} samples")
print(f"Validation: {X_val.shape[0]:,} samples")
print(f"Test:       {X_te.shape[0]:,} samples")

In [ ]:
def build_univariate_lstm(seq_len, n_features=1):
    model = Sequential([
        LSTM(64, input_shape=(seq_len, n_features),
             return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ], name='univariate_lstm')
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse',
                  metrics=['mae'])
    return model

uni_model = build_univariate_lstm(SEQ_LEN)
uni_model.summary()

In [ ]:
callbacks_uni = [
    EarlyStopping(monitor='val_loss', patience=10,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(f'{models}/univariate_lstm.keras',
                    save_best_only=True, monitor='val_loss', verbose=0)
]

print("Training univariate LSTM...")
history_uni = uni_model.fit(
    X_tr,  y_tr,
    validation_data = (X_val, y_val),
    epochs          = 100,
    batch_size      = 64,
    callbacks       = callbacks_uni,
    verbose         = 1
)

print("Training complete!")

In [ ]:
def evaluate_model(model, X_test, y_test, scaler, name):
    y_pred_scaled = model.predict(X_test, verbose=0).flatten()
    y_pred = scaler.inverse_transform(
        y_pred_scaled.reshape(-1,1)).flatten()
    y_true = scaler.inverse_transform(
        y_test.reshape(-1,1)).flatten()

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) /
                          (y_true + 1e-5))) * 100

    print(f"\n{'='*40}")
    print(f"Model: {name}")
    print(f"{'='*40}")
    print(f"  MAE:  €{mae:>15,.0f}")
    print(f"  RMSE: €{rmse:>15,.0f}")
    print(f"  R²:    {r2:>14.4f}")
    print(f"  MAPE:  {mape:>13.2f}%")

    return y_true, y_pred, {'MAE':mae,'RMSE':rmse,'R2':r2,'MAPE':mape}

y_true_uni, y_pred_uni, metrics_uni = evaluate_model(
    uni_model, X_te, y_te, value_scaler, 'Univariate LSTM')

In [ ]:
# ── Merge player features with valuation history ──────────
multi_features = [
    'player_id',
    'age', 'height_in_cm',
    'career_goals', 'career_assists', 'career_apps',
    'goals_per_90', 'assists_per_90',
    'goals_trend_slope', 'assists_trend_slope',
    'consistency_score', 'form_score',
    'total_injuries', 'total_days_out',
    'injury_risk_score', 'chronic_injury_flag',
    'avg_sentiment', 'avg_market_impact',
    'contract_days_remaining',
]

# Keep only columns that exist
multi_features = [c for c in multi_features if c in df_final.columns]
print(f"Using {len(multi_features)} features: {multi_features}")

df_multi = df_val_filtered.merge(
    df_final[multi_features],
    on='player_id', how='left'
).fillna(0)

print(f"\nMultivariate dataset: {df_multi.shape}")
print(df_multi.head(3))

In [ ]:
feature_cols = [c for c in multi_features if c != 'player_id']

feature_scaler = MinMaxScaler()
df_multi[feature_cols] = feature_scaler.fit_transform(
    df_multi[feature_cols])

print(f"Features scaled: {len(feature_cols)} columns")
print(feature_cols)

In [ ]:
def create_multi_sequences(df_player, seq_len, horizon, feat_cols):
    vals     = df_player['value_scaled'].values
    features = df_player[feat_cols].values
    X, y     = [], []
    for i in range(len(vals) - seq_len - horizon + 1):
        # Combine value + features at each timestep
        seq_feats = features[i : i + seq_len]
        seq_val   = vals[i : i + seq_len].reshape(-1, 1)
        X.append(np.hstack([seq_val, seq_feats]))
        y.append(vals[i + seq_len : i + seq_len + horizon])
    return np.array(X), np.array(y)

X_multi, y_multi = [], []

for pid in eligible:
    df_p = df_multi[df_multi['player_id'] == pid]
    if len(df_p) < SEQ_LEN + HORIZON:
        continue
    X_p, y_p = create_multi_sequences(
        df_p, SEQ_LEN, HORIZON, feature_cols)
    if len(X_p) > 0:
        X_multi.append(X_p)
        y_multi.append(y_p)

X_multi = np.vstack(X_multi)
y_multi = np.vstack(y_multi).squeeze()

n_features_multi = X_multi.shape[2]
print(f"\nMultivariate sequences:")
print(f"  X shape: {X_multi.shape}  (samples, timesteps, features)")
print(f"  y shape: {y_multi.shape}")

# Split
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42)
X_tr_m, X_val_m, y_tr_m, y_val_m = train_test_split(
    X_tr_m,  y_tr_m,  test_size=0.1, random_state=42)

print(f"\nTrain:      {X_tr_m.shape[0]:,}")
print(f"Validation: {X_val_m.shape[0]:,}")
print(f"Test:       {X_te_m.shape[0]:,}")

In [ ]:
def build_multivariate_lstm(seq_len, n_features):
    model = Sequential([
        LSTM(128, input_shape=(seq_len, n_features),
             return_sequences=True),
        Dropout(0.3),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1)
    ], name='multivariate_lstm')
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='mse', metrics=['mae'])
    return model

multi_model = build_multivariate_lstm(SEQ_LEN, n_features_multi)
multi_model.summary()

In [ ]:
callbacks_multi = [
    EarlyStopping(monitor='val_loss', patience=10,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(f'{models}/multivariate_lstm.keras',
                    save_best_only=True, monitor='val_loss', verbose=0)
]

print("Training multivariate LSTM...")
history_multi = multi_model.fit(
    X_tr_m,  y_tr_m,
    validation_data = (X_val_m, y_val_m),
    epochs          = 100,
    batch_size      = 64,
    callbacks       = callbacks_multi,
    verbose         = 1
)

print("Training complete!")

y_true_multi, y_pred_multi, metrics_multi = evaluate_model(
    multi_model, X_te_m, y_te_m, value_scaler, 'Multivariate LSTM')

In [ ]:
HORIZON_MULTI = 3   # predict next 3 transfer windows

def create_encoder_decoder_sequences(values, seq_len, horizon):
    X, y = [], []
    for i in range(len(values) - seq_len - horizon + 1):
        X.append(values[i : i + seq_len])
        y.append(values[i + seq_len : i + seq_len + horizon])
    return np.array(X), np.array(y)

X_ed, y_ed = [], []

for pid in eligible:
    vals = df_val_filtered[
        df_val_filtered['player_id'] == pid
    ]['value_scaled'].values

    if len(vals) < SEQ_LEN + HORIZON_MULTI:
        continue

    X_p, y_p = create_encoder_decoder_sequences(
        vals, SEQ_LEN, HORIZON_MULTI)
    if len(X_p) > 0:
        X_ed.append(X_p)
        y_ed.append(y_p)

X_ed = np.vstack(X_ed).reshape(-1, SEQ_LEN, 1)
y_ed = np.vstack(y_ed).reshape(-1, HORIZON_MULTI, 1)

print(f"Encoder-decoder sequences:")
print(f"  X shape: {X_ed.shape}")
print(f"  y shape: {y_ed.shape}")

X_tr_e, X_te_e, y_tr_e, y_te_e = train_test_split(
    X_ed, y_ed, test_size=0.2, random_state=42)
X_tr_e, X_val_e, y_tr_e, y_val_e = train_test_split(
    X_tr_e, y_tr_e, test_size=0.1, random_state=42)

In [ ]:
def build_encoder_decoder(seq_len, horizon, n_features=1):
    inputs  = Input(shape=(seq_len, n_features))

    # Encoder
    enc_out, state_h, state_c = tf.keras.layers.LSTM(
        64, return_state=True, name='encoder')(inputs)

    # RepeatVector — repeat encoder output for each horizon step
    repeated = RepeatVector(horizon)(enc_out)

    # Decoder
    dec_out  = LSTM(64, return_sequences=True,
                    name='decoder')(repeated)
    dec_out  = Dropout(0.2)(dec_out)
    outputs  = TimeDistributed(Dense(1), name='output')(dec_out)

    model = Model(inputs, outputs, name='encoder_decoder_lstm')
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='mse', metrics=['mae'])
    return model

ed_model = build_encoder_decoder(SEQ_LEN, HORIZON_MULTI)
ed_model.summary()

In [ ]:
callbacks_ed = [
    EarlyStopping(monitor='val_loss', patience=10,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(f'{models}/encoder_decoder_lstm.keras',
                    save_best_only=True, monitor='val_loss', verbose=0)
]

print("Training encoder-decoder LSTM...")
history_ed = ed_model.fit(
    X_tr_e, y_tr_e,
    validation_data = (X_val_e, y_val_e),
    epochs          = 100,
    batch_size      = 64,
    callbacks       = callbacks_ed,
    verbose         = 1
)

print("Training complete!")

# Evaluate
y_pred_ed = ed_model.predict(X_te_e, verbose=0)

# Flatten for evaluation
y_pred_ed_flat = value_scaler.inverse_transform(
    y_pred_ed.reshape(-1,1)).flatten()
y_true_ed_flat = value_scaler.inverse_transform(
    y_te_e.reshape(-1,1)).flatten()

mae_ed  = mean_absolute_error(y_true_ed_flat, y_pred_ed_flat)
rmse_ed = np.sqrt(mean_squared_error(y_true_ed_flat, y_pred_ed_flat))
r2_ed   = r2_score(y_true_ed_flat, y_pred_ed_flat)

print(f"\n{'='*40}")
print(f"Model: Encoder-Decoder LSTM")
print(f"{'='*40}")
print(f"  MAE:  €{mae_ed:>15,.0f}")
print(f"  RMSE: €{rmse_ed:>15,.0f}")
print(f"  R²:    {r2_ed:>14.4f}")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('TransferIQ — Milestone 4 LSTM Results', fontsize=15)

# ── Row 1: Univariate ─────────────────────────────────────
axes[0,0].plot(history_uni.history['loss'],     label='Train loss')
axes[0,0].plot(history_uni.history['val_loss'], label='Val loss')
axes[0,0].set_title('Univariate LSTM — loss curves')
axes[0,0].set_xlabel('Epoch')
axes[0,0].set_ylabel('MSE Loss')
axes[0,0].legend()

n = min(200, len(y_true_uni))
axes[0,1].plot(y_true_uni[:n], label='Actual',    alpha=0.8)
axes[0,1].plot(y_pred_uni[:n], label='Predicted', alpha=0.8)
axes[0,1].set_title('Univariate LSTM — predictions vs actual')
axes[0,1].set_xlabel('Sample')
axes[0,1].set_ylabel('Transfer value (EUR)')
axes[0,1].legend()

# ── Row 2: Multivariate ───────────────────────────────────
axes[1,0].plot(history_multi.history['loss'],     label='Train loss')
axes[1,0].plot(history_multi.history['val_loss'], label='Val loss')
axes[1,0].set_title('Multivariate LSTM — loss curves')
axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylabel('MSE Loss')
axes[1,0].legend()

axes[1,1].plot(y_true_multi[:n], label='Actual',    alpha=0.8)
axes[1,1].plot(y_pred_multi[:n], label='Predicted', alpha=0.8)
axes[1,1].set_title('Multivariate LSTM — predictions vs actual')
axes[1,1].set_xlabel('Sample')
axes[1,1].set_ylabel('Transfer value (EUR)')
axes[1,1].legend()

# ── Row 3: Encoder-Decoder ────────────────────────────────
axes[2,0].plot(history_ed.history['loss'],     label='Train loss')
axes[2,0].plot(history_ed.history['val_loss'], label='Val loss')
axes[2,0].set_title('Encoder-Decoder LSTM — loss curves')
axes[2,0].set_xlabel('Epoch')
axes[2,0].set_ylabel('MSE Loss')
axes[2,0].legend()

axes[2,1].plot(y_true_ed_flat[:n], label='Actual',    alpha=0.8)
axes[2,1].plot(y_pred_ed_flat[:n], label='Predicted', alpha=0.8)
axes[2,1].set_title('Encoder-Decoder — multi-step predictions')
axes[2,1].set_xlabel('Sample')
axes[2,1].set_ylabel('Transfer value (EUR)')
axes[2,1].legend()

plt.tight_layout()
plt.savefig(f'{folder}/milestone4_lstm_results.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Loss curves saved!")

In [ ]:
comparison = pd.DataFrame({
    'Model': [
        'Univariate LSTM',
        'Multivariate LSTM',
        'Encoder-Decoder LSTM'
    ],
    'MAE (EUR)': [
        metrics_uni['MAE'],
        metrics_multi['MAE'],
        mae_ed
    ],
    'RMSE (EUR)': [
        metrics_uni['RMSE'],
        metrics_multi['RMSE'],
        rmse_ed
    ],
    'R2 Score': [
        metrics_uni['R2'],
        metrics_multi['R2'],
        r2_ed
    ],
    'MAPE (%)': [
        metrics_uni['MAPE'],
        metrics_multi['MAPE'],
        np.mean(np.abs((y_true_ed_flat - y_pred_ed_flat)
                       / (y_true_ed_flat + 1e-5))) * 100
    ]
})

comparison['MAE (EUR)']  = comparison['MAE (EUR)'].apply( lambda x: f"€{x:,.0f}")
comparison['RMSE (EUR)'] = comparison['RMSE (EUR)'].apply(lambda x: f"€{x:,.0f}")
comparison['R2 Score']   = comparison['R2 Score'].apply(  lambda x: f"{x:.4f}")
comparison['MAPE (%)']   = comparison['MAPE (%)'].apply(  lambda x: f"{x:.2f}%")

print("=== MODEL COMPARISON ===")
print(comparison.to_string(index=False))

comparison.to_csv(f'{folder}/milestone4_model_comparison.csv', index=False)

In [ ]:
# Save all models
uni_model.save(  f'{models}/univariate_lstm.keras')
multi_model.save(f'{models}/multivariate_lstm.keras')
ed_model.save(   f'{models}/encoder_decoder_lstm.keras')

print("All models saved!")

# ── Predict transfer value for a single player ────────────
def predict_player_value(player_id, model, scaler, seq_len):
    vals = df_val_filtered[
        df_val_filtered['player_id'] == player_id
    ]['value_scaled'].values

    if len(vals) < seq_len:
        print(f"Not enough data for player {player_id}")
        return None

    # Use last seq_len values
    seq      = vals[-seq_len:].reshape(1, seq_len, 1)
    pred_sc  = model.predict(seq, verbose=0)[0][0]
    pred_eur = scaler.inverse_transform([[pred_sc]])[0][0]

    actual_eur = scaler.inverse_transform(
        [[vals[-1]]])[0][0]

    print(f"\nPlayer ID:          {player_id}")
    print(f"Current value:      €{actual_eur:>12,.0f}")
    print(f"Predicted value:    €{pred_eur:>12,.0f}")
    print(f"Difference:         €{pred_eur - actual_eur:>+12,.0f}")
    return pred_eur

# Test on first eligible player
sample_pid = list(eligible)[0]
predict_player_value(sample_pid, uni_model, value_scaler, SEQ_LEN)

print("\n=== MILESTONE 4 COMPLETE ===")
print(f"\nSaved files in {models}/:")
for f in os.listdir(models):
    size = os.path.getsize(f'{models}/{f}') / 1024
    print(f"  {f:<40} {size:>8.1f} KB")

In [ ]:
!pip install xgboost lightgbm scikit-learn matplotlib shap --quiet

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap, os, warnings
warnings.filterwarnings('ignore')

import xgboost  as xgb
import lightgbm as lgb

from sklearn.model_selection  import train_test_split, KFold, cross_val_score
from sklearn.preprocessing    import MinMaxScaler
from sklearn.metrics          import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model     import Ridge
import tensorflow as tf

folder = '/content/drive/MyDrive/TransferIQ Project'
models = f'{folder}/models'
os.makedirs(models, exist_ok=True)

print("All libraries ready!")

In [ ]:
df_final = pd.read_csv(f'{folder}/final_feature_set.csv')
df_val   = pd.read_csv(f'{folder}/market_valuations.csv')
df_val['date'] = pd.to_datetime(df_val['date'], errors='coerce')
df_val   = df_val.sort_values(['player_id','date'])

print(f"Final features: {df_final.shape}")
print(f"Valuations:     {df_val.shape}")
print(f"\nColumns in final_feature_set:")
print(df_final.columns.tolist())

In [ ]:
# ── Target variable: market_value_in_eur ─────────────────
TARGET = 'market_value_in_eur'

# ── Select feature columns ────────────────────────────────
drop_cols = [
    'player_id','name','current_club_name',
    'country_of_citizenship','position',
    'foot','age_group','contract_status',
    'performance_trend','form_label',
    'injury_risk_label','value_trend',
    TARGET
]

feature_cols = [c for c in df_final.columns
                if c not in drop_cols
                and df_final[c].dtype in ['float64','int64','uint8']]

print(f"Features selected: {len(feature_cols)}")
print(feature_cols)

# ── Build X and y ─────────────────────────────────────────
df_model = df_final[feature_cols + [TARGET]].dropna(subset=[TARGET])
df_model = df_model[df_model[TARGET] > 0]   # remove zero values

X = df_model[feature_cols].fillna(0).values
y = df_model[TARGET].values

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Target range: €{y.min():,.0f} — €{y.max():,.0f}")
print(f"Target mean:  €{y.mean():,.0f}")

In [ ]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1,1)).flatten()

# Train / validation / test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr, test_size=0.1, random_state=42)

# Also keep unscaled y for reporting
_, y_te_raw  = train_test_split(
    y, test_size=0.2, random_state=42)
y_tr_raw, y_val_raw = train_test_split(
    _, test_size=0.1, random_state=42)

print(f"Train:      {X_tr.shape[0]:,} samples")
print(f"Validation: {X_val.shape[0]:,} samples")
print(f"Test:       {X_te.shape[0]:,} samples")

In [ ]:
def evaluate(name, y_true_raw, y_pred_scaled, scaler):
    y_pred = scaler.inverse_transform(
        y_pred_scaled.reshape(-1,1)).flatten()
    y_true = y_true_raw

    mae   = mean_absolute_error(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    r2    = r2_score(y_true, y_pred)
    mape  = np.mean(np.abs(
        (y_true - y_pred) / (y_true + 1e-5))) * 100

    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(f"  MAE:   €{mae:>15,.0f}")
    print(f"  RMSE:  €{rmse:>15,.0f}")
    print(f"  R²:     {r2:>14.4f}")
    print(f"  MAPE:   {mape:>13.2f}%")

    return y_pred, {'name':name,'MAE':mae,'RMSE':rmse,'R2':r2,'MAPE':mape}

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = 42,
    early_stopping_rounds = 20,
    eval_metric       = 'rmse',
    verbosity         = 0,
)

print("Training XGBoost...")
xgb_model.fit(
    X_tr,  y_tr,
    eval_set        = [(X_val, y_val)],
    verbose         = 50,
)

xgb_pred_te, metrics_xgb = evaluate(
    'XGBoost', y_te_raw,
    xgb_model.predict(X_te), scaler_y)

# Save
xgb_model.save_model(f'{models}/xgboost_model.json')
print("\nXGBoost saved!")

In [ ]:
lgb_model = lgb.LGBMRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    num_leaves        = 50,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = 42,
    verbose           = -1,
)

print("Training LightGBM...")
lgb_model.fit(
    X_tr,  y_tr,
    eval_set        = [(X_val, y_val)],
    callbacks       = [lgb.early_stopping(20, verbose=False),
                       lgb.log_evaluation(50)],
)

lgb_pred_te, metrics_lgb = evaluate(
    'LightGBM', y_te_raw,
    lgb_model.predict(X_te), scaler_y)

# Save
lgb_model.booster_.save_model(f'{models}/lightgbm_model.txt')
print("\nLightGBM saved!")

In [ ]:
# Load trained LSTM models from Milestone 4
uni_model   = tf.keras.models.load_model(
    f'{models}/univariate_lstm.keras')
multi_model = tf.keras.models.load_model(
    f'{models}/multivariate_lstm.keras')

# Get LSTM predictions on the same test players
df_val_filtered = df_val.copy()
player_counts   = df_val.groupby('player_id')['market_value_in_eur'].count()
eligible        = player_counts[player_counts >= 8].index

value_scaler = MinMaxScaler()
df_val_filtered['value_scaled'] = value_scaler.fit_transform(
    df_val_filtered[['market_value_in_eur']])

SEQ_LEN = 6

# Build univariate sequences for LSTM predictions
def create_sequences(values, seq_len=6):
    X = []
    for i in range(len(values) - seq_len):
        X.append(values[i: i + seq_len])
    return np.array(X)

lstm_preds_dict = {}

for pid in eligible:
    vals = df_val_filtered[
        df_val_filtered['player_id'] == pid
    ]['value_scaled'].values

    if len(vals) < SEQ_LEN + 1:
        continue

    seq     = vals[-SEQ_LEN:].reshape(1, SEQ_LEN, 1)
    pred_sc = uni_model.predict(seq, verbose=0)[0][0]
    pred_eur= value_scaler.inverse_transform([[pred_sc]])[0][0]
    lstm_preds_dict[pid] = pred_eur

print(f"LSTM predictions generated for {len(lstm_preds_dict):,} players")

# Add LSTM prediction as feature for ensemble
df_final['lstm_pred'] = df_final['player_id'].map(lstm_preds_dict).fillna(0)
print("LSTM predictions added to feature set!")